In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByMerge: 814,255 chars, 47 unbalanced classes
#  Classes: 0-9 (10) + A-Z merged with a-z where visually similar (47 total)
#  Merged pairs (uppercase kept): C/c, I/i, J/j, K/k, L/l, M/m, O/o,
#                                  P/p, S/s, U/u, V/v, W/w, X/x, Y/y, Z/z
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/bymerge"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 60
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05        # smoothing helps with unbalanced classes
DROP_PATH    = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/bymerge"
AUTOTUNE     = tf.data.AUTOTUNE

# tfds ByMerge label order: digits 0-9, then letters in merged order
# (exact mapping available in tfds metadata; indices follow NIST conventions)
LABELS = (
    [str(d) for d in range(10)]
    + ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
       'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
       'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't']
)  # 47 labels

os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds    = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

class DropPath(keras.layers.Layer):
    def __init__(self, drop_prob=0.0, **kwargs):
        super().__init__(**kwargs); self.drop_prob = drop_prob
    def call(self, x, training=None):
        if not training or self.drop_prob == 0.0: return x
        keep  = 1.0 - self.drop_prob
        shape = (tf.shape(x)[0],) + (1,) * (len(x.shape) - 1)
        mask  = tf.floor(keep + tf.random.uniform(shape, dtype=x.dtype))
        return x * mask / keep
    def get_config(self):
        cfg = super().get_config(); cfg["drop_prob"] = self.drop_prob; return cfg

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels, drop_rate=0.0):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    if drop_rate > 0: x = DropPath(drop_rate)(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch, drop_rate=0.0):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch, drop_rate)
    r2  = residual_block(r1, out_ch, drop_rate)
    r3  = residual_block(r2, out_ch, drop_rate)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def english_topology_module(x, out_features):
    """Symmetric kernels + dilated convs for Latin strokes."""
    h  = layers.Conv2D(48, (1, 5), padding="same", use_bias=False, activation=gelu)(x)
    v  = layers.Conv2D(48, (5, 1), padding="same", use_bias=False, activation=gelu)(x)
    d1 = layers.Conv2D(32, 3, padding="same", dilation_rate=1, use_bias=False, activation=gelu)(x)
    d2 = layers.Conv2D(32, 3, padding="same", dilation_rate=2, use_bias=False, activation=gelu)(x)
    d4 = layers.Conv2D(32, 3, padding="same", dilation_rate=4, use_bias=False, activation=gelu)(x)
    out = layers.Concatenate()([h, v, d1, d2, d4])
    out = layers.BatchNormalization()(out)
    out = layers.GlobalAveragePooling2D()(out)
    return layers.Dense(out_features, activation=gelu)(out)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL  —  slightly narrower than ByClass (fewer output classes)
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=47, image_size=28, drop_path_rate=0.10):
    K, DR = num_classes, drop_path_rate
    inp = Input(shape=(image_size, image_size, 1), name="input")

    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    enc1 = dense_res_block(stem, 64,  64,  drop_rate=DR * 0.5)
    enc2 = dense_res_block(enc1, 64,  128, drop_rate=DR)
    enc3 = dense_res_block(enc2, 128, 256, drop_rate=DR)

    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    multi_scale = layers.Concatenate()([gap1, gap2, gap3])

    stm_out = english_topology_module(enc3, 256)

    fused = layers.Concatenate()([stm_out, multi_scale])
    x   = layers.Dense(256, use_bias=False)(fused)
    x   = layers.LayerNormalization()(x)
    x   = layers.Activation(gelu)(x)
    x   = layers.Dropout(0.25)(x)
    out = layers.Dense(K, name="logits")(x)
    return Model(inputs=inp, outputs=out, name="WhatNet_ByMerge")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG, DROP_PATH)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

# ─────────────────────────────────────────────────────────────────────────────
#  SAVE
# ─────────────────────────────────────────────────────────────────────────────
results = {
    "dataset": "EMNIST_BYMERGE", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

2026-04-22 08:07:21.825023: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776845242.016608      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776845242.071138      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776845242.481867      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776845242.481901      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776845242.481903      55 computation_placer.cc:177] computation placer alr

[INFO] Loading emnist/bymerge via tensorflow_datasets ...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/emnist/bymerge/incomplete.HBRPOI_3.1.0/emnist-test.tfrecord*...:   0%|    …

Dataset emnist downloaded and prepared to /root/tensorflow_datasets/emnist/bymerge/3.1.0. Subsequent calls will reuse this data.
[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Steps/epoch: 9814
[INFO] Params: 5,685,247
Epoch 1/60


I0000 00:00:1776845749.262015     132 service.cc:152] XLA service 0x78aff002e000 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776845749.262105     132 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1776845749.262112     132 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1776845753.606882     132 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1776845778.027862     132 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


9815/9815 ━━━━━━━━━━━━━━━━━━━━ 867s 83ms/step - accuracy: 0.3895 - loss: 2.5291 - val_accuracy: 0.8607 - val_loss: 0.7680
Epoch 2/60
9815/9815 ━━━━━━━━━━━━━━━━━━━━ 775s 79ms/step - accuracy: 0.8548 - loss: 0.7855 - val_accuracy: 0.8641 - val_loss: 0.7317
Epoch 3/60
4317/9815 ━━━━━━━━━━━━━━━━━━━━ 6:55 76ms/step - accuracy: 0.8687 - loss: 0.7324

# Method

In [ ]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByMerge: 814,255 chars, 47 UNBALANCED classes
#  Classes: digits 0-9 + letters where visually similar cases are merged.
#  Merged pairs (uppercase kept): C/c, I/i, J/j, K/k, L/l, M/m, O/o,
#                                  P/p, S/s, U/u, V/v, W/w, X/x, Y/y, Z/z
#  Key challenge: severe class imbalance — macro F1 matters more than accuracy.
#  Horizontal flip ENABLED — mixed digit+letter classes, flip helps letters
#  but we accept minor digit confusion as trade-off for letter regularisation.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/bymerge"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 1
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10          # unbalanced data + merging → higher smoothing
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/bymerge"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = (
    [str(d) for d in range(10)]
    + ['A','B','C','D','E','F','G','H','I','J','K','L','M',
       'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
       'a','b','d','e','f','g','h','n','q','r','t']
)  # 47 labels
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS  —  compute from a label scan to counter imbalance
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights from training labels ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_raw.take(n_train):
    label_counts[int(label.numpy())] += 1

# Balanced inverse-frequency weighting, clipped to avoid extreme weights
total_samples  = label_counts.sum()
class_weights  = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights  = np.clip(class_weights, 0.1, 10.0)   # clip outliers
class_weight_dict = {i: float(class_weights[i]) for i in range(NUM_CLASSES)}
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)   # helps minority letter classes
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 — unbalanced 47-class task benefits from richer routing
    to avoid the capsule collapsing onto majority-class features.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=47, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    # ── Multi-scale GAP fusion ────────────────────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    # ── Adaptive Filter Capsule ───────────────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(256, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion ──────────────────────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_ByMerge")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight_dict,          # counter class imbalance
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYMERGE", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

# Method

In [ ]:
import os, random, json
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG  —  EMNIST ByMerge: 814,255 chars, 47 UNBALANCED classes
#  Classes: digits 0-9 + letters where visually similar cases are merged.
#  Merged pairs (uppercase kept): C/c, I/i, J/j, K/k, L/l, M/m, O/o,
#                                  P/p, S/s, U/u, V/v, W/w, X/x, Y/y, Z/z
#  Key challenge: severe class imbalance — macro F1 matters more than accuracy.
#  Horizontal flip ENABLED — mixed digit+letter classes, flip helps letters
#  but we accept minor digit confusion as trade-off for letter regularisation.
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "emnist/bymerge"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 1
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10          # unbalanced data + merging → higher smoothing
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/bymerge"
AUTOTUNE     = tf.data.AUTOTUNE

LABELS = (
    [str(d) for d in range(10)]
    + ['A','B','C','D','E','F','G','H','I','J','K','L','M',
       'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
       'a','b','d','e','f','g','h','n','q','r','t']
)  # 47 labels
os.makedirs(RESULTS_DIR, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} via tensorflow_datasets ...")
train_raw, info = tfds.load(
    DATASET, split="train", as_supervised=True, with_info=True, shuffle_files=True,
)
test_raw = tfds.load(DATASET, split="test", as_supervised=True)

total   = info.splits["train"].num_examples
n_val   = int(total * VAL_SPLIT)
n_train = total - n_val
print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {info.splits['test'].num_examples:,}")

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS  —  compute from a label scan to counter imbalance
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights from training labels ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_raw.take(n_train):
    label_counts[int(label.numpy())] += 1

# Balanced inverse-frequency weighting, clipped to avoid extreme weights
total_samples  = label_counts.sum()
class_weights  = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights  = np.clip(class_weights, 0.1, 10.0)   # clip outliers
class_weight_dict = {i: float(class_weights[i]) for i in range(NUM_CLASSES)}
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

# ─────────────────────────────────────────────────────────────────────────────
#  PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    label = tf.cast(label, tf.int32)
    return image, label

def augment(image, label):
    image = tf.image.random_brightness(image, 0.20)
    image = tf.image.random_contrast(image, 0.80, 1.20)
    image = tf.pad(image, [[2, 2], [2, 2], [0, 0]], constant_values=-1.0)
    image = tf.image.random_crop(image, [IMG, IMG, 1])
    image = tf.image.random_flip_left_right(image)   # helps minority letter classes
    image = image + tf.random.normal(tf.shape(image), stddev=0.02)
    image = tf.clip_by_value(image, -1.0, 1.0)
    return image, label

def to_onehot(image, label):
    return image, tf.one_hot(label, NUM_CLASSES)

train_raw = train_raw.shuffle(20_000, seed=SEED, reshuffle_each_iteration=False)
val_raw   = train_raw.take(n_val)
train_raw = train_raw.skip(n_val)

train_ds = (
    train_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(augment,    num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
val_ds = (
    val_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BS).prefetch(AUTOTUNE)
test_ds_oh = (
    test_raw
    .map(preprocess, num_parallel_calls=AUTOTUNE)
    .map(to_onehot,  num_parallel_calls=AUTOTUNE)
    .batch(BS).prefetch(AUTOTUNE)
)

steps_per_epoch = n_train // BS
total_steps     = steps_per_epoch * EPOCHS
print(f"[INFO] Steps/epoch: {steps_per_epoch}")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x): return tf.nn.gelu(x)

def channel_attention(x, channels, reduction=4):
    gap  = layers.GlobalAveragePooling2D(keepdims=True)(x)
    gap  = layers.Reshape((channels,))(gap)
    attn = layers.Dense(max(1, channels // reduction), activation="relu")(gap)
    attn = layers.Dense(channels, activation="sigmoid")(attn)
    attn = layers.Reshape((1, 1, channels))(attn)
    return layers.Multiply()([x, attn])

def residual_block(x, channels):
    r = x
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Add()([x, r]); x = layers.Activation(gelu)(x)
    return x

def dense_res_block(x, in_ch, out_ch):
    if in_ch != out_ch:
        x = layers.Conv2D(out_ch, 1, use_bias=False)(x)
        x = layers.BatchNormalization()(x); x = layers.Activation(gelu)(x)
    r1  = residual_block(x,  out_ch)
    r2  = residual_block(r1, out_ch)
    r3  = residual_block(r2, out_ch)
    cat = layers.Concatenate()([r1, r2, r3])
    out = layers.Conv2D(out_ch, 1, use_bias=False)(cat)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    out = layers.DepthwiseConv2D(3, strides=2, padding="same", use_bias=False)(out)
    out = layers.Conv2D(out_ch, 1, use_bias=False)(out)
    out = layers.BatchNormalization()(out); out = layers.Activation(gelu)(out)
    return out

def adaptive_filter_capsule(x, num_classes, capsule_dim=32):
    """
    Lightweight capsule routing.
    capsule_dim=32 — unbalanced 47-class task benefits from richer routing
    to avoid the capsule collapsing onto majority-class features.
    """
    h = layers.Dense(256, activation=gelu)(x)
    h = layers.Dense(num_classes * capsule_dim)(h)
    h = layers.Reshape((num_classes, capsule_dim))(h)
    x_exp    = layers.RepeatVector(num_classes)(x)
    x_sliced = layers.Lambda(lambda t: t[:, :, :capsule_dim])(x_exp)
    caps = layers.Multiply()([x_sliced, h])
    caps = layers.Lambda(lambda t: tf.reduce_sum(t, axis=-1))(caps)
    return layers.BatchNormalization()(caps)

# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
def build_model(num_classes=47, image_size=28):
    K   = num_classes
    inp = Input(shape=(image_size, image_size, 1), name="input")

    # ── Stem ──────────────────────────────────────────────────────────────
    t = layers.Conv2D(32, 3, padding="same", use_bias=False)(inp)
    t = layers.BatchNormalization()(t); t = layers.Activation(gelu)(t)
    h = layers.Conv2D(16, (1, 5), padding="same", use_bias=False)(inp)
    h = layers.BatchNormalization()(h); h = layers.Activation(gelu)(h)
    v = layers.Conv2D(16, (5, 1), padding="same", use_bias=False)(inp)
    v = layers.BatchNormalization()(v); v = layers.Activation(gelu)(v)

    stem = layers.Concatenate()([t, h, v])
    stem = channel_attention(stem, 64)
    stem = layers.Conv2D(64, 1, use_bias=False)(stem)
    stem = layers.BatchNormalization()(stem); stem = layers.Activation(gelu)(stem)

    # ── Encoder ───────────────────────────────────────────────────────────
    enc1 = dense_res_block(stem, 64,  64)
    enc2 = dense_res_block(enc1, 64,  128)
    enc3 = dense_res_block(enc2, 128, 256)

    # ── Multi-scale GAP fusion ────────────────────────────────────────────
    gap1 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc1))
    gap2 = layers.Dense(128, activation=gelu)(layers.GlobalAveragePooling2D()(enc2))
    gap3 = layers.GlobalAveragePooling2D()(enc3)
    fused_gap = layers.Concatenate(name="multiscale_fused")([gap1, gap2, gap3])  # (B, 512)

    # ── Adaptive Filter Capsule ───────────────────────────────────────────
    afc_out = adaptive_filter_capsule(fused_gap, K, capsule_dim=32)   # (B, K)

    # ── Dense head ────────────────────────────────────────────────────────
    x = layers.Dense(256, use_bias=False, name="head_dense")(fused_gap)
    x = layers.LayerNormalization(name="head_ln")(x)
    x = layers.Activation(gelu, name="head_act")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(K, name="head_logits")(x)

    # ── Gated fusion ──────────────────────────────────────────────────────
    combined   = layers.Concatenate(name="gate_input")([x, afc_out])
    gate       = layers.Dense(2, activation="softmax", name="gate")(combined)
    x_scaled   = layers.Lambda(
        lambda t: t[0] * t[1][:, 0:1], name="gate_dense")([x, gate])
    afc_scaled = layers.Lambda(
        lambda t: t[0] * t[1][:, 1:2], name="gate_afc")([afc_out, gate])
    out = layers.Add(name="logits")([x_scaled, afc_scaled])

    return Model(inputs=inp, outputs=out, name="WhatNet_ByMerge")

# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE
# ─────────────────────────────────────────────────────────────────────────────
class WarmupCosineDecay(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, base, total_steps, warmup_steps):
        self.base         = base
        self.total_steps  = tf.cast(total_steps,  tf.float32)
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup_lr = self.base * step / tf.maximum(self.warmup_steps, 1.0)
        progress  = (step - self.warmup_steps) / tf.maximum(
            self.total_steps - self.warmup_steps, 1.0)
        cosine_lr = tf.maximum(self.base * 0.5 * (1.0 + tf.cos(np.pi * progress)), 1e-6)
        return tf.where(step < self.warmup_steps, warmup_lr, cosine_lr)
    def get_config(self):
        return {"base": self.base, "total_steps": int(self.total_steps),
                "warmup_steps": int(self.warmup_steps)}

# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model        = build_model(NUM_CLASSES, IMG)
warmup_steps = steps_per_epoch * WARMUP_EP
lr_sch       = WarmupCosineDecay(LR, total_steps, warmup_steps)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=lr_sch, weight_decay=WEIGHT_DECAY),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True, label_smoothing=LABEL_SMOOTH),
    metrics=["accuracy"],
    jit_compile=True,
)
print(f"[INFO] Params: {model.count_params():,}")

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    class_weight=class_weight_dict,          # counter class imbalance
    callbacks=[
        ModelCheckpoint(os.path.join(RESULTS_DIR, "best_model.keras"),
                        monitor="val_accuracy", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_accuracy", patience=12,
                      restore_best_weights=True, verbose=1),
    ],
)

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)
test_acc *= 100.0

tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)

for images, labels in test_ds:
    preds = tf.argmax(model(images, training=False), axis=1).numpy()
    lbls  = labels.numpy()
    for c in range(NUM_CLASSES):
        tp[c] += np.sum((preds == c) & (lbls == c))
        fp[c] += np.sum((preds == c) & (lbls != c))
        fn[c] += np.sum((preds != c) & (lbls == c))
        correct_per_class[c] += np.sum(preds[lbls == c] == c)
        total_per_class[c]   += np.sum(lbls == c)

prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {model.count_params():,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYMERGE", "num_classes": NUM_CLASSES,
    "test_acc": round(test_acc, 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4), "params": model.count_params(),
    "per_class_acc": {LABELS[c]: round(per_class_acc[c], 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.history.items()}, f, indent=2)
print(f"\n[INFO] Saved to {RESULTS_DIR}/"); print("[DONE]")

In [ ]:
import os, random, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split
from torchvision import datasets, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import math

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─────────────────────────────────────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATASET      = "EMNIST ByMerge"
NUM_CLASSES  = 47
IMG          = 28
BS           = 64
EPOCHS       = 30
LR           = 5e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.10
WARMUP_EP    = 5
VAL_SPLIT    = 0.10
RESULTS_DIR  = "./results/bymerge"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

LABELS = (
    [str(d) for d in range(10)]
    + ['A','B','C','D','E','F','G','H','I','J','K','L','M',
       'N','O','P','Q','R','S','T','U','V','W','X','Y','Z',
       'a','b','d','e','f','g','h','n','q','r','t']
)  # 47 labels

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"[INFO] Device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  TRANSFORMS
#  EMNIST images are transposed vs standard — apply fix via rotate+flip.
# ─────────────────────────────────────────────────────────────────────────────
emnist_fix = transforms.Compose([
    transforms.Lambda(lambda img: transforms.functional.rotate(img, -90)),
    transforms.Lambda(lambda img: transforms.functional.hflip(img)),
])

train_transform = transforms.Compose([
    emnist_fix,
    transforms.RandomAffine(degrees=0, translate=(0.07, 0.07)),   # ~2px pad+crop equiv
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),                                          # [0,1]
    transforms.Normalize((0.5,), (0.5,)),                          # → [-1,1]
    transforms.Lambda(lambda x: x + torch.randn_like(x) * 0.02),  # Gaussian noise
    transforms.Lambda(lambda x: x.clamp(-1.0, 1.0)),
])

val_transform = transforms.Compose([
    emnist_fix,
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

# ─────────────────────────────────────────────────────────────────────────────
#  DATASETS
# ─────────────────────────────────────────────────────────────────────────────
print(f"[INFO] Loading {DATASET} ...")
full_train = datasets.EMNIST(root="./data", split="bymerge", train=True,
                              download=True, transform=train_transform)
test_ds    = datasets.EMNIST(root="./data", split="bymerge", train=False,
                              download=True, transform=val_transform)

n_val   = int(len(full_train) * VAL_SPLIT)
n_train = len(full_train) - n_val
train_subset, val_subset = random_split(
    full_train, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
# val subset should use val_transform — wrap it
val_subset.dataset = datasets.EMNIST(root="./data", split="bymerge", train=True,
                                      download=False, transform=val_transform)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds):,}")

train_loader = DataLoader(train_subset, batch_size=BS, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_subset,   batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,      batch_size=BS, shuffle=False,
                          num_workers=4, pin_memory=True)

# ─────────────────────────────────────────────────────────────────────────────
#  CLASS WEIGHTS
# ─────────────────────────────────────────────────────────────────────────────
print("[INFO] Computing class weights ...")
label_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, label in train_subset:
    label_counts[int(label)] += 1

total_samples = label_counts.sum()
class_weights = total_samples / (NUM_CLASSES * np.maximum(label_counts, 1))
class_weights = np.clip(class_weights, 0.1, 10.0)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
print(f"[INFO] Class weight range: [{class_weights.min():.2f}, {class_weights.max():.2f}]")

# ─────────────────────────────────────────────────────────────────────────────
#  BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────
def gelu(x):
    return F.gelu(x)


class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc1 = nn.Linear(channels, max(1, channels // reduction))
        self.fc2 = nn.Linear(max(1, channels // reduction), channels)

    def forward(self, x):
        b, c, _, _ = x.shape
        attn = self.gap(x).view(b, c)
        attn = F.relu(self.fc1(attn))
        attn = torch.sigmoid(self.fc2(attn)).view(b, c, 1, 1)
        return x * attn


class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        r = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        return F.gelu(x + r)


class DenseResBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.proj = None
        if in_ch != out_ch:
            self.proj = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, bias=False),
                nn.BatchNorm2d(out_ch),
            )
        self.r1 = ResidualBlock(out_ch)
        self.r2 = ResidualBlock(out_ch)
        self.r3 = ResidualBlock(out_ch)
        # concat of 3 residual blocks → 3*out_ch → out_ch
        self.fuse = nn.Sequential(
            nn.Conv2d(3 * out_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        # Depthwise strided downsampling
        self.dw   = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1,
                              groups=out_ch, bias=False)
        self.pw   = nn.Conv2d(out_ch, out_ch, 1, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))
        r1  = self.r1(x)
        r2  = self.r2(r1)
        r3  = self.r3(r2)
        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.fuse(cat))
        out = F.gelu(self.bn(self.pw(self.dw(out))))
        return out


class AdaptiveFilterCapsule(nn.Module):
    """Lightweight capsule routing (capsule_dim=32)."""
    def __init__(self, in_features, num_classes, capsule_dim=32):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1  = nn.Linear(in_features, 256)
        self.fc2  = nn.Linear(256, num_classes * capsule_dim)
        self.bn   = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        h     = F.gelu(self.fc1(x))
        h     = self.fc2(h).view(-1, self.num_classes, self.capsule_dim)  # (B, K, D)
        x_exp = x.unsqueeze(1).expand(-1, self.num_classes, -1)           # (B, K, F)
        x_sl  = x_exp[:, :, :self.capsule_dim]                            # (B, K, D)
        caps  = (x_sl * h).sum(dim=-1)                                    # (B, K)
        return self.bn(caps)


# ─────────────────────────────────────────────────────────────────────────────
#  MODEL
# ─────────────────────────────────────────────────────────────────────────────
class WhatNetByMerge(nn.Module):
    def __init__(self, num_classes=47, image_size=28):
        super().__init__()
        K = num_classes

        # ── Stem ──────────────────────────────────────────────────────────
        self.stem_t = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32))
        self.stem_h = nn.Sequential(
            nn.Conv2d(1, 16, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(16))
        self.stem_v = nn.Sequential(
            nn.Conv2d(1, 16, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(16))
        self.stem_ca  = ChannelAttention(64)
        self.stem_proj = nn.Sequential(
            nn.Conv2d(64, 64, 1, bias=False), nn.BatchNorm2d(64))

        # ── Encoder ───────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(64,  64)
        self.enc2 = DenseResBlock(64,  128)
        self.enc3 = DenseResBlock(128, 256)

        # ── Multi-scale GAP projections ───────────────────────────────────
        self.gap_proj1 = nn.Linear(64,  128)
        self.gap_proj2 = nn.Linear(128, 128)
        # enc3 goes directly (256 features); fused = 128+128+256 = 512

        # ── Capsule ───────────────────────────────────────────────────────
        self.afc = AdaptiveFilterCapsule(512, K, capsule_dim=32)

        # ── Dense head ────────────────────────────────────────────────────
        self.head_dense  = nn.Linear(512, 256, bias=False)
        self.head_ln     = nn.LayerNorm(256)
        self.head_drop   = nn.Dropout(0.25)
        self.head_logits = nn.Linear(256, K)

        # ── Gate ──────────────────────────────────────────────────────────
        self.gate = nn.Linear(2 * K, 2)

    def forward(self, x):
        # Stem
        t    = F.gelu(self.stem_t(x))
        h    = F.gelu(self.stem_h(x))
        v    = F.gelu(self.stem_v(x))
        stem = torch.cat([t, h, v], dim=1)
        stem = self.stem_ca(stem)
        stem = F.gelu(self.stem_proj(stem))

        # Encoder
        e1 = self.enc1(stem)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)

        # Multi-scale GAP
        g1 = F.gelu(self.gap_proj1(e1.mean(dim=[2, 3])))
        g2 = F.gelu(self.gap_proj2(e2.mean(dim=[2, 3])))
        g3 = e3.mean(dim=[2, 3])
        fused = torch.cat([g1, g2, g3], dim=1)  # (B, 512)

        # Capsule
        afc_out = self.afc(fused)

        # Dense head
        x = F.gelu(self.head_ln(self.head_dense(fused)))
        x = self.head_drop(x)
        x = self.head_logits(x)

        # Gated fusion
        gate_w   = F.softmax(self.gate(torch.cat([x, afc_out], dim=1)), dim=1)
        x_scaled = x       * gate_w[:, 0:1]
        a_scaled = afc_out * gate_w[:, 1:2]
        return x_scaled + a_scaled


# ─────────────────────────────────────────────────────────────────────────────
#  LR SCHEDULE  (warmup + cosine decay)
# ─────────────────────────────────────────────────────────────────────────────
def make_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / max(warmup_steps, 1)
        progress = (current_step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return max(0.5 * (1.0 + math.cos(math.pi * progress)), 1e-6 / LR)
    return LambdaLR(optimizer, lr_lambda)


# ─────────────────────────────────────────────────────────────────────────────
#  LOSS  (label smoothing + per-sample class weighting)
# ─────────────────────────────────────────────────────────────────────────────
class WeightedLabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes, smoothing, class_weights):
        super().__init__()
        self.smoothing      = smoothing
        self.num_classes    = num_classes
        self.class_weights  = class_weights  # (K,) tensor on device

    def forward(self, logits, targets):
        log_probs  = F.log_softmax(logits, dim=-1)
        smooth_val = self.smoothing / self.num_classes
        # Build smooth target distribution
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs, smooth_val)
            smooth_targets.scatter_(1, targets.unsqueeze(1),
                                    1.0 - self.smoothing + smooth_val)
        loss        = -(smooth_targets * log_probs).sum(dim=-1)          # (B,)
        sample_wts  = self.class_weights[targets]                        # (B,)
        return (loss * sample_wts).mean()


# ─────────────────────────────────────────────────────────────────────────────
#  TRAIN
# ─────────────────────────────────────────────────────────────────────────────
model = WhatNetByMerge(NUM_CLASSES, IMG).to(DEVICE)
print(f"[INFO] Params: {sum(p.numel() for p in model.parameters()):,}")

steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * EPOCHS
warmup_steps    = steps_per_epoch * WARMUP_EP

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler   = make_warmup_cosine_scheduler(optimizer, total_steps, warmup_steps)
criterion   = WeightedLabelSmoothingLoss(NUM_CLASSES, LABEL_SMOOTH, class_weights_tensor)

best_val_acc  = 0.0
best_state    = None
patience      = 12
no_improve    = 0
history       = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

print(f"[INFO] Steps/epoch: {steps_per_epoch} | Total: {total_steps}")

for epoch in range(1, EPOCHS + 1):
    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    run_loss, run_correct, run_total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        run_loss    += loss.item() * images.size(0)
        run_correct += (logits.argmax(1) == labels).sum().item()
        run_total   += images.size(0)

    train_loss = run_loss / run_total
    train_acc  = run_correct / run_total

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits  = model(images)
            loss    = criterion(logits, labels)
            val_loss    += loss.item() * images.size(0)
            val_correct += (logits.argmax(1) == labels).sum().item()
            val_total   += images.size(0)

    val_loss /= val_total
    val_acc   = val_correct / val_total

    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    print(f"Epoch {epoch:>3}/{EPOCHS} | "
          f"loss {train_loss:.4f}  acc {train_acc*100:.2f}% | "
          f"val_loss {val_loss:.4f}  val_acc {val_acc*100:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, os.path.join(RESULTS_DIR, "best_model.pt"))
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"[INFO] Early stopping at epoch {epoch}")
            break

# Restore best weights
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

# ─────────────────────────────────────────────────────────────────────────────
#  EVALUATE
# ─────────────────────────────────────────────────────────────────────────────
model.eval()
tp = np.zeros(NUM_CLASSES); fp = np.zeros(NUM_CLASSES); fn = np.zeros(NUM_CLASSES)
correct_per_class = np.zeros(NUM_CLASSES); total_per_class = np.zeros(NUM_CLASSES)
test_loss_total, test_total = 0.0, 0

ce_loss = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=LABEL_SMOOTH)

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss   = ce_loss(logits, labels)
        test_loss_total += loss.item() * images.size(0)
        test_total      += images.size(0)

        preds = logits.argmax(1).cpu().numpy()
        lbls  = labels.cpu().numpy()
        for c in range(NUM_CLASSES):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))
            correct_per_class[c] += np.sum(preds[lbls == c] == c)
            total_per_class[c]   += np.sum(lbls == c)

test_loss     = test_loss_total / test_total
test_acc      = correct_per_class.sum() / total_per_class.sum() * 100.0
prec          = tp / (tp + fp + 1e-8)
rec           = tp / (tp + fn + 1e-8)
f1            = 2 * prec * rec / (prec + rec + 1e-8)
macro_f1      = float(f1.mean() * 100.0)
per_class_acc = correct_per_class / np.maximum(total_per_class, 1) * 100.0
worst5_idx    = np.argsort(per_class_acc)[:5]

print(f"\n[RESULT] Test Acc : {test_acc:.2f}%")
print(f"[RESULT] Macro F1 : {macro_f1:.2f}%")
print(f"[RESULT] Test Loss: {test_loss:.4f}")
print(f"[RESULT] Params   : {sum(p.numel() for p in model.parameters()):,}")
print(f"\n[RESULT] Worst-5 classes:")
for idx in worst5_idx:
    print(f"         '{LABELS[idx]}' (cls {idx:>2}) = {per_class_acc[idx]:.1f}%")

results = {
    "dataset": "EMNIST_BYMERGE", "num_classes": NUM_CLASSES,
    "test_acc": round(float(test_acc), 2), "macro_f1": round(macro_f1, 2),
    "test_loss": round(float(test_loss), 4),
    "params": sum(p.numel() for p in model.parameters()),
    "per_class_acc": {LABELS[c]: round(float(per_class_acc[c]), 2) for c in range(NUM_CLASSES)},
}
with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)
with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump({k: [float(v) for v in vs] for k, vs in history.items()}, f, indent=2)

print(f"\n[INFO] Saved to {RESULTS_DIR}/")
print("[DONE]")

# Output

In [ ]:
[INFO] Device: cuda
[INFO] Loading EMNIST ByMerge ...
100%|██████████| 562M/562M [00:03<00:00, 181MB/s]  
[INFO] Train: 628,139 | Val: 69,793 | Test: 116,323
[INFO] Computing class weights ...
[INFO] Class weight range: [0.39, 5.86]
[INFO] Params: 5,735,419
[INFO] Steps/epoch: 9815 | Total: 981500
Epoch   1/100 | loss 1.7377  acc 65.40% | val_loss 1.1357  val_acc 84.93%
Epoch   2/100 | loss 1.1332  acc 82.97% | val_loss 1.0974  val_acc 83.29%
Epoch   3/100 | loss 1.0645  acc 84.74% | val_loss 1.0338  val_acc 84.74%
Epoch   4/100 | loss 1.0289  acc 85.59% | val_loss 1.0086  val_acc 85.29%
Epoch   5/100 | loss 1.0071  acc 86.16% | val_loss 0.9980  val_acc 86.01%
Epoch   6/100 | loss 0.9862  acc 86.75% | val_loss 0.9724  val_acc 88.30%
Epoch   7/100 | loss 0.9690  acc 87.30% | val_loss 0.9683  val_acc 87.28%
Epoch   8/100 | loss 0.9563  acc 87.68% | val_loss 0.9514  val_acc 88.42%
Epoch   9/100 | loss 0.9472  acc 87.94% | val_loss 0.9443  val_acc 88.93%
Epoch  10/100 | loss 0.9397  acc 88.16% | val_loss 0.9436  val_acc 88.37%
Epoch  11/100 | loss 0.9332  acc 88.36% | val_loss 0.9386  val_acc 88.26%
Epoch  12/100 | loss 0.9275  acc 88.53% | val_loss 0.9385  val_acc 88.38%
Epoch  13/100 | loss 0.9223  acc 88.69% | val_loss 0.9350  val_acc 88.78%
Epoch  14/100 | loss 0.9180  acc 88.80% | val_loss 0.9273  val_acc 88.88%
Epoch  15/100 | loss 0.9136  acc 88.96% | val_loss 0.9324  val_acc 88.32%
Epoch  16/100 | loss 0.9099  acc 89.07% | val_loss 0.9223  val_acc 88.82%
Epoch  17/100 | loss 0.9067  acc 89.17% | val_loss 0.9296  val_acc 88.83%
Epoch  18/100 | loss 0.9026  acc 89.29% | val_loss 0.9252  val_acc 89.48%
Epoch  19/100 | loss 0.8993  acc 89.43% | val_loss 0.9242  val_acc 88.58%
Epoch  20/100 | loss 0.8956  acc 89.55% | val_loss 0.9225  val_acc 89.35%
Epoch  21/100 | loss 0.8925  acc 89.60% | val_loss 0.9253  val_acc 88.72%
Epoch  22/100 | loss 0.8891  acc 89.72% | val_loss 0.9312  val_acc 88.91%
Epoch  23/100 | loss 0.8864  acc 89.80% | val_loss 0.9210  val_acc 89.39%
Epoch  24/100 | loss 0.8831  acc 89.91% | val_loss 0.9269  val_acc 89.56%
Epoch  25/100 | loss 0.8800  acc 90.01% | val_loss 0.9272  val_acc 89.40%
Epoch  26/100 | loss 0.8771  acc 90.10% | val_loss 0.9378  val_acc 89.62%
Epoch  27/100 | loss 0.8740  acc 90.20% | val_loss 0.9334  val_acc 89.70%
Epoch  28/100 | loss 0.8710  acc 90.35% | val_loss 0.9317  val_acc 88.81%
Epoch  29/100 | loss 0.8678  acc 90.39% | val_loss 0.9338  val_acc 89.67%
Epoch  30/100 | loss 0.8651  acc 90.46% | val_loss 0.9428  val_acc 89.35%
Epoch  31/100 | loss 0.8623  acc 90.57% | val_loss 0.9364  val_acc 89.25%
Epoch  32/100 | loss 0.8591  acc 90.70% | val_loss 0.9355  val_acc 89.68%
Epoch  33/100 | loss 0.8557  acc 90.84% | val_loss 0.9457  val_acc 89.76%
Epoch  34/100 | loss 0.8530  acc 90.90% | val_loss 0.9410  val_acc 89.66%
Epoch  35/100 | loss 0.8495  acc 91.03% | val_loss 0.9493  val_acc 89.20%
Epoch  36/100 | loss 0.8469  acc 91.16% | val_loss 0.9571  val_acc 89.68%
Epoch  37/100 | loss 0.8439  acc 91.20% | val_loss 0.9543  val_acc 89.66%
Epoch  38/100 | loss 0.8406  acc 91.37% | val_loss 0.9475  val_acc 89.53%
Epoch  39/100 | loss 0.8376  acc 91.44% | val_loss 0.9637  val_acc 89.41%
Epoch  40/100 | loss 0.8348  acc 91.55% | val_loss 0.9633  val_acc 89.69%
Epoch  41/100 | loss 0.8323  acc 91.61% | val_loss 0.9721  val_acc 89.31%
Epoch  42/100 | loss 0.8295  acc 91.80% | val_loss 0.9613  val_acc 89.50%
Epoch  43/100 | loss 0.8264  acc 91.91% | val_loss 0.9704  val_acc 89.50%
Epoch  44/100 | loss 0.8239  acc 91.99% | val_loss 0.9774  val_acc 89.53%
Epoch  45/100 | loss 0.8211  acc 92.07% | val_loss 0.9731  val_acc 89.60%
[INFO] Early stopping at epoch 45

[RESULT] Test Acc : 89.71%
[RESULT] Macro F1 : 88.76%
[RESULT] Test Loss: 1.5491
[RESULT] Params   : 5,735,419

[RESULT] Worst-5 classes:
         'L' (cls 21) = 51.3%
         'f' (cls 40) = 59.6%
         'q' (cls 44) = 60.8%
         'I' (cls 18) = 64.5%
         'O' (cls 24) = 68.3%

[INFO] Saved to ./results/bymerge/
[DONE]